In [ ]:
import os
import json
import math


input_file = "../data/mimic_cxr_annotaion_json/chat_test_MIMIC_CXR_all_gpt4extract_rulebased_v1.json";
out_file = "../data/mimic_cxr_annotaion_json/chat_test_MIMIC_CXR_all_gpt4extract_rulebased_v1_rag_indexing.json";
def map_label(val):
    if val == 0:
        return '0'
    elif val == 1:
        return '1'
    elif val == -1:
        return '2'
    elif val is None or (isinstance(val, float) and math.isnan(val)):
        return '2'
    else:
        return '2'

# JSON 파일 로딩
with open(input_file, "r") as f:
    data = json.load(f)

# 중복 이미지 제거 및 변환 처리
unique_images = set()
converted = []

for item in data:
    image_path = item["image"]
    if image_path in unique_images:
        continue
    unique_images.add(image_path)

    # 라벨 변환
    chexpert_labels = item.get("chexpert_labels", {})
    mapped_labels = ''.join(map_label(chexpert_labels.get(k)) for k in sorted(chexpert_labels))
    item["chexpert_labels_mapped"] = mapped_labels

    # txt 경로 추출
    base_dir = os.path.dirname(image_path)
    study_id = base_dir.split("/")[-1]
    txt_file_path = base_dir + f"/{study_id}.txt"
    # txt_file_path = "../data/mimic_cxr_annotaion_json/" + txt_file_path;

    # try:
    #     with open(txt_file_path, "r", encoding="utf-8") as f:
    #         txt_content = f.read().strip()
    # except FileNotFoundError:
    #     txt_content = None  # 없을 경우 None

    item["txt_report"] = txt_file_path
    converted.append(item)

# 결과 저장
with open(out_file, "w", encoding="utf-8") as f:
    json.dump(converted, f, indent=2, ensure_ascii=False)

# 이미지 개수 출력
print(f"총 {len(unique_images)}개의 고유 이미지가 처리되었습니다.")


총 4827개의 고유 이미지가 처리되었습니다.


In [3]:
import json
from collections import Counter
out_file = "../data/mimic_cxr_annotaion_json/chat_test_MIMIC_CXR_all_gpt4extract_rulebased_v1_rag_indexing.json";

# JSON 파일 열기
with open(out_file, "r", encoding="utf-8") as f:
    data = json.load(f)

# txt_report 값 수집
chexpert_labels_mapped = []

for item in data:
    txt = item.get("chexpert_labels_mapped", None)
    # None은 문자열로 'None' 처리해서 포함
    chexpert_labels_mapped.append(txt if txt is not None else "None")

# 등장 횟수 집계
counter = Counter(chexpert_labels_mapped)

# 결과 출력
print("chexpert_labels_mapped 값별 개수:")
for txt, count in counter.most_common():
    print(f"[{txt[:50]}{'...' if len(txt) > 50 else ''}] → {count}개")

print(f"\n총 고유 chexpert_labels_mapped 값 종류 수: {len(counter)}")

chexpert_labels_mapped 값별 개수:
[22222222122222] → 611개
[22222221222222] → 227개
[22222222222222] → 161개
[22212222222222] → 113개
[22222222222122] → 88개
[22222222212222] → 87개
[22222222122221] → 85개
[21222222222222] → 75개
[12222222212222] → 67개
[22222222122022] → 64개
[22212222212222] → 61개
[22222221212222] → 57개
[12222222222222] → 55개
[22222221222122] → 53개
[21212222222222] → 51개
[22222221222221] → 46개
[22222222122201] → 42개
[22122222222222] → 30개
[12222221222222] → 26개
[21212222212222] → 26개
[21222222222221] → 26개
[12222221212222] → 25개
[22222222212221] → 25개
[12222222212221] → 25개
[22222211222222] → 24개
[22222221212221] → 23개
[21212222212201] → 22개
[22212221222222] → 22개
[22222122222222] → 21개
[22222222212202] → 20개
[22212222222221] → 20개
[22022222222222] → 20개
[22212221212222] → 20개
[12222222222221] → 19개
[21222221222222] → 19개
[11222222212221] → 19개
[22122222222122] → 18개
[21202222202022] → 18개
[22222212222222] → 18개
[12222221212201] → 17개
[12212221212222] → 17개
[21222222212221] → 16개


In [2]:
import os
import json

# JSON 경로
json_path = "../data/mimic_cxr_annotaion_json/chat_test_MIMIC_CXR_all_gpt4extract_rulebased_v1_rag_indexing.json";
base_path = "/workspace/"  # 실제 파일 시스템 기준 루트 경로 (상황에 맞게 수정)

# JSON 열기
with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

missing_images = []
missing_txts = []

for item in data:
    image_rel = item.get("image")
    image_path = os.path.join(base_path, image_rel)

    # 이미지 파일 존재 확인
    if not os.path.exists(image_path):
        missing_images.append(image_path)

    # txt 경로 추정
    base_dir = os.path.dirname(image_path)
    study_id = base_dir.split("/")[-1]
    txt_path = os.path.join(base_dir, f"{study_id}.txt")

    # 텍스트 파일 존재 확인
    if not os.path.exists(txt_path):
        missing_txts.append(txt_path)

# 결과 출력
print(f"존재하지 않는 이미지 파일 수: {len(missing_images)}")
print(f"존재하지 않는 텍스트 파일 수: {len(missing_txts)}")

# (선택) 누락된 파일 경로 출력
# if missing_images:
#     print("\n[누락된 이미지 파일]")
#     for p in missing_images[:5]:  # 너무 많으면 일부만
#         print(p)

# if missing_txts:
#     print("\n[누락된 텍스트 파일]")
#     for p in missing_txts[:5]:
#         print(p)


존재하지 않는 이미지 파일 수: 0
존재하지 않는 텍스트 파일 수: 0


In [11]:
import os
import json

input_file = "/workspace/LLaVA-Med-RAG/data/test/llava_med_instruct_1k_mimic_cxr_train_test_convert.json"
out_file = "/workspace/LLaVA-Med-RAG/data/test/llava_med_instruct_1k_mimic_cxr_train_test_convert_rag.json"

def map_label(val):
    if val == "0":
        return '0'
    elif val == "1":
        return '1'
    elif val == "-1":
        return '2'
    elif val == "NaN" or val is None:
        return '2'
    else:
        return '2'

# JSON Lines 읽기
data = []
with open(input_file, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            data.append(json.loads(line))

# 중복 이미지 제거 및 처리
unique_images = set()
converted = []

for item in data:
    image_path = item["image"]
    if image_path in unique_images:
        continue
    unique_images.add(image_path)

    # chexpert_labels 추출
    chexpert_labels = item.get("in_text_mention", {}).get("chexpert_labels", {})
    # 잘못된 키 제거 (빈 문자열)
    chexpert_labels = {k: v for k, v in chexpert_labels.items() if k.strip() != ""}
    # 정렬 및 매핑
    mapped_labels = ''.join(map_label(str(chexpert_labels.get(k, "NaN"))) for k in sorted(chexpert_labels))

    item["chexpert_labels_mapped"] = mapped_labels

    # txt 경로 추가
    base_dir = os.path.dirname(image_path)
    study_id = base_dir.split("/")[-1]
    txt_file_path = base_dir + f"/{study_id}.txt"
    item["txt_report"] = txt_file_path

    converted.append(item)

# 결과 저장 (jsonl)
with open(out_file, "w", encoding="utf-8") as f:
    for item in converted:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"총 {len(unique_images)}개의 고유 이미지가 처리되었습니다.")


총 584개의 고유 이미지가 처리되었습니다.


In [7]:
!pip install pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 268.2 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 14.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 509.2/509.2 kB 5.4 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 347.8/347.8 kB 3.6 MB/s eta 0:00:00ta 0:00:01

[notice] A new release of pip is available: 23.3.1 -> 25.0.1
[notice] To update, run: python -m pip install --upgrade pip


In [10]:
import json
import math
import pandas as pd  # NaN 처리를 위해 사용

# NaN 처리를 위해 기본 설정
NaN = float("nan")


input_file = "/workspace/LLaVA-Med-RAG/data/llava_med_instruct_1k_mimic_cxr_train_test.json"
output_file = "/workspace/LLaVA-Med-RAG/data/test/llava_med_instruct_1k_mimic_cxr_train_test_convert.json"

# 원본 데이터 로딩
with open(input_file, "r", encoding="utf-8") as f:
    original_data = json.load(f)

converted = []
for idx, item in enumerate(original_data):
    qid = f"{idx+1}_{item['id']}"
    image = item.get("image")
    
    conversations = item.get("conversations", [])
    text = next((c["value"] for c in conversations if c["from"] == "human"), "")
    gpt4_answer = next((c["value"] for c in conversations if c["from"] == "gpt"), "")

    view = item.get("view", "")
    orientation = item.get("orientation", "")
    chexpert_labels = item.get("chexpert_labels", {})

    # NaN 문자열로 변환
    chexpert_labels_str = {
        k: "NaN" if (v is None or (isinstance(v, float) and math.isnan(v))) else str(int(v)) if isinstance(v, float) and not math.isnan(v) else str(v)
        for k, v in chexpert_labels.items()
    }

    converted.append({
        "question_id": qid,
        "image": image,
        "text": text,
        "gpt4_answer": gpt4_answer,
        "fig_caption": item.get("reason", ""),
        "in_text_mention": {
            "view": view,
            "orientation": orientation,
            "chexpert_labels": chexpert_labels_str
        }
    })

# 변환된 데이터 저장
with open(output_file, "w", encoding="utf-8") as f:
    for item in converted:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"✅ {len(converted)}개 항목이 변환되어 saved to 'converted.jsonl'")


✅ 1000개 항목이 변환되어 saved to 'converted.jsonl'
